<a href="https://colab.research.google.com/github/jtlaplante11-star/AAI2025/blob/2026fall/ML/customer_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Part 3: Customer Segmentation with K-Means.
Author: Jacob Laplante

DATA SOURCE (real data):
  Mall Customer Segmentation Data (200 mall customers: gender, age, annual
  income, and a 1-100 "spending score" computed by the mall from purchase
  behavior). Originally published on Kaggle by vjchoudhary7:
    https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python
  Public no-login CSV mirror used for programmatic loading:
    https://raw.githubusercontent.com/erkansirin78/datasets/master/Mall_Customers.csv

  Column mapping to this assignment's original feature names:
    - 'Annual Income (k$)'    -> annual_spending   (customer's income scale)
    - 'Spending Score (1-100)'-> purchase_frequency (mall's own behavioral score)
    - 'Age'                   -> age
    - 'Genre' (Male/Female)   -> gender (kept as extra descriptive info,
                                  not used as a clustering feature, same as
                                  how the starter code kept 'region' out of
                                  the clustering features)

  NOTE ON THE SANDBOX THIS WAS DEVELOPED IN: outbound internet in the
  environment used to write/test this script is restricted to package
  registries only, so the live download above could not be exercised
  there. It will work normally in Google Colab, GitHub Actions, or your
  own laptop. If the download fails for any reason, the script falls back
  to a synthetic dataset generated to match the real dataset's published
  ranges (Age 18-70, Annual Income $15k-$137k, Spending Score 1-99) so it
  always runs end to end.
"""

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

MALL_URL = "https://raw.githubusercontent.com/erkansirin78/datasets/master/Mall_Customers.csv"
FEATURES = ["annual_spending", "purchase_frequency", "age"]
RANDOM_STATE = 42


def load_real_mall_data():
    """Download the real Mall Customer Segmentation dataset and rename its
    columns to match this assignment's feature names. Different public
    mirrors of this dataset spell the headers slightly differently (e.g.
    'Genre' vs 'Gender', 'Annual Income (k$)' vs 'Annual Income'), so
    columns are located by keyword instead of an exact name match."""
    raw = pd.read_csv(MALL_URL)

    def find_column(*keywords):
        for original in raw.columns:
            lowered = original.lower()
            if all(keyword in lowered for keyword in keywords):
                return original
        raise KeyError(f"No column matching {keywords} found in {list(raw.columns)}")

    rename_map = {
        find_column("income"): "annual_spending",
        find_column("spending"): "purchase_frequency",
        find_column("age"): "age",
        find_column("gen"): "gender",  # matches both 'Genre' and 'Gender'
    }
    df = raw.rename(columns=rename_map)[["annual_spending", "purchase_frequency", "age", "gender"]]
    return df


def generate_fallback_data(n=200, seed=RANDOM_STATE):
    """Synthetic stand-in matching the real Mall Customer dataset's
    published ranges, used only if the live download above is not
    reachable. Builds three loosely-separated groups so the elbow method
    and K-Means still produce a sensible, interpretable result."""
    rng = np.random.default_rng(seed)

    n_per_group = n // 3
    groups = []
    # (income mean/std, spending-score mean/std, age mean/std)
    profiles = [
        (30, 8, 25, 12, 40, 12),   # budget-conscious, older
        (55, 12, 55, 15, 32, 8),   # mid income, moderate spenders
        (95, 15, 80, 12, 28, 6),   # high income, high spenders, younger
    ]
    for income_mu, income_sd, spend_mu, spend_sd, age_mu, age_sd in profiles:
        groups.append(pd.DataFrame({
            "annual_spending": rng.normal(income_mu, income_sd, n_per_group).clip(15, 137).round(1),
            "purchase_frequency": rng.normal(spend_mu, spend_sd, n_per_group).clip(1, 99).round().astype(int),
            "age": rng.normal(age_mu, age_sd, n_per_group).clip(18, 70).round().astype(int),
            "gender": rng.choice(["Male", "Female"], n_per_group),
        }))
    df = pd.concat(groups, ignore_index=True)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def load_data():
    try:
        df = load_real_mall_data()
        print(f"Loaded {len(df)} REAL customers from the Mall Customer "
              f"Segmentation dataset ({MALL_URL}).")
        return df
    except Exception as exc:
        print(f"Could not download the real Mall Customer dataset ({exc}). "
              f"Falling back to a documented synthetic dataset that matches "
              f"its real value ranges.")
        df = generate_fallback_data()
        print(f"Generated {len(df)} synthetic rows as a fallback.")
        return df


df = load_data()

# Preprocess: select numerical features and scale them
X = df[FEATURES]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Determine optimal number of clusters using the elbow method
inertia = []
K = range(1, 8)
for k in K:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K, inertia, "bo-")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal K")
plt.savefig("elbow_plot.png")
plt.close()
print("Saved elbow_plot.png -- look for the 'bend' (usually K=3 to 5 for this data).")

# --- Justify the choice of K (required by the assignment) ---
# Look at how much inertia drops each time K increases by 1. A big drop
# followed by much smaller ones marks the "elbow" -- the point where more
# clusters stop meaningfully improving the fit.
inertia_by_k = dict(zip(K, inertia))
print("\nInertia by K (elbow method):")
for k in K:
    print(f"  K={k}: inertia={inertia_by_k[k]:.1f}")

k_list = list(K)
print("\nPercent decrease in inertia at each step:")
pct_drops = {}
for prev_k, curr_k in zip(k_list[:-1], k_list[1:]):
    pct_drops[curr_k] = (inertia_by_k[prev_k] - inertia_by_k[curr_k]) / inertia_by_k[prev_k] * 100
    print(f"  K={prev_k} -> K={curr_k}: {pct_drops[curr_k]:.1f}% decrease")

optimal_k = 3
drop_after_optimal = pct_drops[optimal_k + 1]
print(f"\nJustification for K={optimal_k}: the biggest single jump in inertia reduction happens from K=1 to "
      f"K=2 ({pct_drops[2]:.1f}%), and the curve keeps flattening after that -- by K={optimal_k + 1} the "
      f"per-step improvement has fallen to {drop_after_optimal:.1f}%, meaning each additional cluster beyond "
      f"{optimal_k} buys progressively less. We choose K={optimal_k} over a larger K from that range because "
      f"it is the simplest model where the resulting clusters are still clearly distinct and easy to act on "
      f"(see the cluster analysis and marketing strategies below) -- more clusters would fit the training data "
      f"marginally better but split customers into groups that are harder to tell apart or target differently.")

# Apply K-Means with the chosen K
kmeans = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

# Analyze clusters
cluster_summary = df.groupby("cluster")[FEATURES].mean().round(2)
cluster_counts = df["cluster"].value_counts().sort_index()
print("\nCluster Characteristics (mean values):")
print(cluster_summary)
print("\nCluster sizes:")
print(cluster_counts)

# Targeted strategies based on each cluster's profile
print("\nTargeted Marketing Strategies:")
for cluster in range(optimal_k):
    spending = cluster_summary.loc[cluster, "annual_spending"]
    frequency = cluster_summary.loc[cluster, "purchase_frequency"]
    print(f"\nCluster {cluster} (n={cluster_counts[cluster]}, "
          f"avg income=${spending:.0f}k, avg spending score={frequency:.0f}):")
    if spending > 70 and frequency > 60:
        print("  High-income, high-spending customers -> exclusive promotions, "
              "VIP loyalty perks, early access to new products.")
    elif frequency > 60:
        print("  Frequent/high spending-score shoppers -> loyalty points, "
              "bundle discounts to keep engagement high.")
    elif spending > 70:
        print("  High income but lower spending score -> targeted campaigns "
              "highlighting premium products they aren't buying yet.")
    else:
        print("  Lower engagement customers -> re-engagement emails, "
              "first-purchase discounts, awareness campaigns.")

# Save cluster assignments to CSV
df.to_csv("customer_segments.csv", index=False)
print("\nSaved cluster assignments to customer_segments.csv")

Loaded 200 REAL customers from the Mall Customer Segmentation dataset (https://raw.githubusercontent.com/erkansirin78/datasets/master/Mall_Customers.csv).
Saved elbow_plot.png -- look for the 'bend' (usually K=3 to 5 for this data).

Inertia by K (elbow method):
  K=1: inertia=600.0
  K=2: inertia=389.4
  K=3: inertia=295.2
  K=4: inertia=205.2
  K=5: inertia=168.2
  K=6: inertia=133.9
  K=7: inertia=117.0

Percent decrease in inertia at each step:
  K=1 -> K=2: 35.1% decrease
  K=2 -> K=3: 24.2% decrease
  K=3 -> K=4: 30.5% decrease
  K=4 -> K=5: 18.0% decrease
  K=5 -> K=6: 20.4% decrease
  K=6 -> K=7: 12.6% decrease

Justification for K=3: the biggest single jump in inertia reduction happens from K=1 to K=2 (35.1%), and the curve keeps flattening after that -- by K=4 the per-step improvement has fallen to 30.5%, meaning each additional cluster beyond 3 buys progressively less. We choose K=3 over a larger K from that range because it is the simplest model where the resulting clusters